# Machine Learning Upskilling: Regression & Sklearn
---
**Duration:** 4 Hours
**Target Audience:** Data Analysts transitioning to Data Science
**Dataset:** King County House Sales (Kaggle)

This workshop covers the end-to-end supervised machine learning workflow for regression. We will build a pipeline handling over 80 features and 21,000+ rows, moving beyond simple model fitting to emphasize robust preprocessing, cross-validation, and statistical assumption checking.

In [ ]:
# Import necessary libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Sklearn modules
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Statsmodels for inference
import statsmodels.api as sm
import scipy.stats as stats

import warnings
warnings.filterwarnings('ignore')

## Hour 1: The Sklearn Ecosystem & Preprocessing
### 1.1 Load the Dataset
We will use the Kaggle King County House Sales dataset. Our goal is to predict the `price` of a house based on its attributes (21,613 rows, 21 initial columns).

In [ ]:
# Load data directly from a public repository
url = "https://raw.githubusercontent.com/Shreyas3108/house-price-prediction/master/kc_house_data.csv"
df = pd.read_csv(url)

print(f"Dataset Shape: {df.shape}")
df.head()

### 1.2 Feature Engineering & Splitting
Data Analysts know that raw data is rarely ready for modeling. Let's extract the year and month from the date column, drop identifiers, and split our data.

In [ ]:
# Convert date to datetime and extract useful numerical features
df['date'] = pd.to_datetime(df['date'])
df['year_sold'] = df['date'].dt.year
df['month_sold'] = df['date'].dt.month

# Drop the original date and the unique ID (which has no predictive power)
df = df.drop(columns=['id', 'date'])

# Separate Target (y) and Features (X)
X = df.drop('price', axis=1)
y = df['price']

print(f"Features Shape before categorical encoding: {X.shape}")

# 80/20 Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

### 1.3 Building the Preprocessing Pipeline
We have continuous data (like `sqft_living`) and categorical data (like `zipcode`). We must treat them differently using a `ColumnTransformer`.

In [ ]:
# Define which columns are categorical vs numerical
categorical_features = ['zipcode', 'waterfront']
numeric_features = [col for col in X.columns if col not in categorical_features]

# Create a preprocessing transformer
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_features),
        ('cat', OneHotEncoder(handle_unknown='ignore', drop='first'), categorical_features)
    ])

print("Preprocessor defined successfully! OneHotEncoding zipcodes will drastically expand our feature space.")

## Hour 2: Regression Fundamentals & Model Fitting
Linear regression fits a line to our multidimensional data. Let's add the predictor to our pipeline and fit it to our training data.

In [ ]:
# Create the full modeling pipeline
pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', LinearRegression())
])

# Fit the model on the training data
pipeline.fit(X_train, y_train)

# Generate predictions for the test set
y_pred = pipeline.predict(X_test)

print("Model fitted and predictions generated!")

## Hour 3: Robust Evaluation & Testing
### 3.1 Basic Metrics
How wrong are our predictions? We use MAE, RMSE, and R-Squared.

In [ ]:
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print(f"Mean Absolute Error: ${mae:,.2f}")
print(f"Root Mean Squared Error: ${rmse:,.2f}")
print(f"R-Squared: {r2:.4f}")

### 3.2 K-Fold Cross Validation
A single train-test split can be lucky or unlucky. K-Fold Cross Validation splits the data $K$ times to give us a robust average error score across different slices of the data.

In [ ]:
# Perform 5-fold cross validation on the entire dataset
cv_scores = cross_val_score(pipeline, X, y, cv=5, scoring='neg_root_mean_squared_error')

# Convert negative RMSE back to positive
cv_rmse = -cv_scores

print(f"Cross-Validation RMSE scores: {cv_rmse}")
print(f"Average CV RMSE: ${cv_rmse.mean():,.2f} (+/- ${cv_rmse.std():,.2f})")

## Hour 4: Assumption Checks & Statistical Inference
Machine Learning focuses on predicting (Sklearn), but Statistics focuses on explaining (Statsmodels). Before trusting our explanation, we must check our assumptions.

### 4.1 Normality of Residuals
Residuals (Errors) should be normally distributed. We check this visually using a Histogram and a Q-Q Plot.

In [ ]:
residuals = y_test - y_pred

fig, ax = plt.subplots(1, 2, figsize=(12, 5))

# Histogram
sns.histplot(residuals, kde=True, ax=ax[0])
ax[0].set_title('Histogram of Residuals')

# Q-Q Plot
stats.probplot(residuals, dist="norm", plot=ax[1])
ax[1].set_title('Q-Q Plot')

plt.tight_layout()
plt.show()

### 4.2 Homoscedasticity
The variance of our errors should be constant. We plot Predictions vs Residuals. A funnel shape indicates Heteroscedasticity (a violation!).

In [ ]:
plt.figure(figsize=(8, 5))
sns.scatterplot(x=y_pred, y=residuals, alpha=0.3)
plt.axhline(y=0, color='r', linestyle='--')
plt.xlabel('Predicted Values')
plt.ylabel('Residuals')
plt.title('Residuals vs Predicted Values (Checking Homoscedasticity)')
plt.show()

### 4.3 Statistical Summary using Statsmodels
To explain *why* houses are priced the way they are, we use `statsmodels`. Because our Pipeline encoded 70 zipcodes into new columns, we need to extract the new feature names first!

In [ ]:
# 1. Preprocess the training data manually to pass into statsmodels
X_train_transformed = preprocessor.fit_transform(X_train)

# Convert sparse matrix to dense array (caused by OneHotEncoder)
if hasattr(X_train_transformed, "toarray"):
    X_train_transformed = X_train_transformed.toarray()

# 2. Extract the dynamically generated feature names
cat_names = preprocessor.named_transformers_['cat'].get_feature_names_out(categorical_features)
all_feature_names = numeric_features + list(cat_names)

# 3. Statsmodels requires adding a constant (intercept) explicitly
X_train_sm = sm.add_constant(X_train_transformed)
sm_feature_names = ['const'] + all_feature_names

# 4. Fit OLS model and print summary
ols_model = sm.OLS(y_train.values, X_train_sm).fit()
print(ols_model.summary(xname=sm_feature_names))